In [ ]:
# Supplementary Table S5
# APOD glycopeptiforms tested for trans colocalization at the GRN locus (chr17),
# with glycan annotation and colocalization status with the AD GWAS (Bellenguez).
suppressMessages({library(colocboost); library(data.table); library(tidyverse)})

anno <- fread('rosmap/glycoproteomics/dlpfcTissue/batch1/annotation/gpAnnotation.csv', sep = ',')
APOD <- readRDS('Work/gpQTL/data_output/trans/colocboost/with_HF/APOD/transboost/transboost_AD.chr17_ENSG00000030582.tb_xqtl_AD_Bellenguez.rds')

# tested APOD glycopeptiforms (outcomes of the colocboost object)
on <- APOD[[1]]$data_info$outcome_info$outcome_names
tested_gp <- unique(na.omit(str_extract(on, 'gp_\\d+')))

# robust colocalization signals (CoS)
# fA <- get_robust_colocalization(APOD[[1]], cos_npc_cutoff = 0.5, npc_outcome_cutoff = 0.3)
fA <- get_robust_colocalization(APOD[[1]], cos_npc_cutoff = 0.2, npc_outcome_cutoff = 0.2, pvalue_cutoff = 1e-4)

cs <- fA$cos_summary

In [ ]:

# Raw CoS membership before robust postprocessing.
robust_cos_ids <- sub(":.*$", "", cs$cos_id)
raw_cos_outcomes <- APOD[[1]]$cos_details$cos_outcomes$outcome_name
raw_cos_variables <- APOD[[1]]$cos_details$cos$cos_variables
raw_cos_npc <- APOD[[1]]$cos_details$cos_npc
raw_min_npc_outcome <- APOD[[1]]$cos_details$cos_min_npc_outcome
coloc_variables <- APOD[[1]]$data_info$variables
coloc_coef <- APOD[[1]]$data_info$coef
coloc_z <- APOD[[1]]$data_info$z
raw_gp_cos <- rbindlist(lapply(seq_along(raw_cos_outcomes), function(i) {
  raw_cos_id <- sub(":.*$", "", names(raw_cos_outcomes)[i])
  gps <- unique(na.omit(str_extract(raw_cos_outcomes[[i]], 'gp_\\d+')))
  if (length(gps) == 0) return(NULL)
  variant_index <- match(raw_cos_variables[[i]], coloc_variables)
  stopifnot(!anyNA(variant_index))
  rbindlist(lapply(gps, function(gp) {
    outcome_name <- raw_cos_outcomes[[i]][str_detect(raw_cos_outcomes[[i]], paste0('_', gp, '$'))]
    stopifnot(length(outcome_name) == 1L)
    z_values <- coloc_z[[outcome_name]][variant_index]
    effect_sizes <- coloc_coef[[outcome_name]][variant_index]
    p_values <- 2 * pnorm(-abs(z_values))
    stopifnot(all(is.finite(p_values)), all(is.finite(effect_sizes)))
    data.table(
      gpID = gp,
      raw_cos_id = raw_cos_id,
      raw_cos_variants = paste(raw_cos_variables[[i]], collapse = ', '),
      raw_variant_pvalues = paste(formatC(p_values, format = 'e', digits = 5), collapse = ', '),
      raw_variant_effect_sizes = paste(formatC(effect_sizes, format = 'e', digits = 5), collapse = ', '),
      raw_cos_npc = round(as.numeric(raw_cos_npc[i]), 3),
      raw_min_npc_outcome = round(as.numeric(raw_min_npc_outcome[i]), 3),
      status = ifelse(
        raw_cos_id %in% robust_cos_ids,
        'retained',
        'filtered out (not all outcomes met p-value < 1e-4)'
      )
    )
  }))
}))

# Per-glycopeptiform robust CoS membership after postprocessing.
gp_cos <- rbindlist(lapply(seq_len(nrow(cs)), function(i){
  outs <- str_split(cs$colocalized_outcomes[i], '; ')[[1]]
  gps  <- unique(na.omit(str_extract(outs, 'gp_\\d+')))
  if (length(gps) == 0) return(NULL)
  data.table(gpID = gps,
             cos_id = sub(":.*$", "", cs$cos_id[i]),
             is_ad   = cs$focal_outcome[i] == 'AD_Bellenguez',
             signal  = ifelse(cs$focal_outcome[i] == 'AD_Bellenguez', 'AD-colocalized', 'glycopeptiform-only'),
             lead    = cs$top_variable[i],
             cos_npc = round(as.numeric(cs$cos_npc[i]), 3))
}))


In [ ]:

raw_gp_sum <- raw_gp_cos[, .(
  raw_cos_id = paste(unique(raw_cos_id), collapse = '; '),
  raw_cos_variants = paste(paste0(raw_cos_id, ': ', raw_cos_variants), collapse = ' | '),
  raw_variant_pvalues = paste(paste0(raw_cos_id, ': ', raw_variant_pvalues), collapse = ' | '),
  raw_variant_effect_sizes = paste(paste0(raw_cos_id, ': ', raw_variant_effect_sizes), collapse = ' | '),
  raw_cos_npc = paste(unique(raw_cos_npc), collapse = '; '),
  raw_min_npc_outcome = paste(unique(raw_min_npc_outcome), collapse = '; '),
  postprocessing_status = paste(
    unique(paste0(raw_cos_id, ': ', status)), collapse = '; '
  )
), by = gpID]

gp_sum <- gp_cos[, .(cos_id = paste(unique(cos_id), collapse = '; '), 
                     ad_coloc = ifelse(any(is_ad), 'Yes', 'No'),
                     coloc_signal = paste(sort(unique(signal)), collapse = '; '),
                     coloc_lead_variant = paste(unique(lead), collapse = '; '),
                     cos_npc = paste(unique(cos_npc), collapse = '; ')),
                 by = gpID]

# annotation for the 30 tested APOD glycopeptiforms (lean: no gene coordinates / redundant IDs)
tab <- unique(anno[gpID %in% tested_gp,
                   .(gpID, GlycoSite, Sequence, GlycanComposition, GlycanClass)])
tab <- merge(tab, raw_gp_sum, by = 'gpID', all.x = TRUE)
tab <- merge(tab, gp_sum, by = 'gpID', all.x = TRUE)
tab[is.na(postprocessing_status), postprocessing_status := 'No raw CoS']
tab[is.na(ad_coloc), ad_coloc := 'No']
tab[is.na(coloc_signal), coloc_signal := 'none']
tab[is.na(coloc_lead_variant), coloc_lead_variant := '']
tab[is.na(cos_npc), cos_npc := '']

filtered_cos1_gp <- c('gp_1807', 'gp_1811')
stopifnot(
  setequal(
    tab[grepl('cos1: filtered out', postprocessing_status, fixed = TRUE), gpID],
    filtered_cos1_gp
  ),
  all(tab[gpID %in% filtered_cos1_gp, raw_cos_id] == 'cos1'),
  all(is.na(tab[gpID %in% filtered_cos1_gp, cos_id]))
)

# order: AD-colocalized first, then by glycopeptiform number
tab[, gp_num := as.integer(str_extract(gpID, '\\d+'))]
setorder(tab, -ad_coloc, gp_num)
tab[, gp_num := NULL]
tab[, gene_name := "APOD"]

In [ ]:

# Retain only publication-facing annotation and robust colocalization results.
# Raw CoS fields remain available above for validation but are not exported.
filtered_cos1_rows <- sum(grepl(
  'cos1: filtered out', tab$postprocessing_status, fixed = TRUE
))
publication_cols <- c(
  'gene_name', 'gpID', 'GlycoSite', 'Sequence', 'GlycanComposition', 'GlycanClass',
  'cos_id', 'ad_coloc', 'coloc_signal', 'coloc_lead_variant', 'cos_npc'
)
tab <- tab[, ..publication_cols]
setnames(tab, c(
  'Gene', 'Glycopeptiform ID', 'Glycosite', 'Peptide sequence',
  'Glycan composition', 'Glycan class', 'Robust CoS ID', 'AD colocalized',
  'Colocalization signal', 'Colocalization lead variant',
  'Colocalization confidence (NPC)'
))

table_note <- paste0(
  'Note: Colocalization set 1 (cos1) was excluded during postprocessing because not all outcomes met ',
  'the nominal association threshold (P < 1 × 10⁻⁴).'
)
note_row <- as.data.table(setNames(as.list(rep('', ncol(tab))), names(tab)))
note_row[[1]] <- table_note
table_s3 <- rbindlist(list(tab, note_row), use.names = TRUE)

fwrite(table_s3, 'TableS5.csv', na = '')
cat(
  'wrote TableS5.csv  data rows:', nrow(tab),
  ' output rows including note:', nrow(table_s3),
  ' columns:', ncol(table_s3),
  ' AD-colocalized:', sum(tab$`AD colocalized` == 'Yes'),
  ' filtered cos1 rows verified:', filtered_cos1_rows,
  '\n'
)


In [ ]:
tab

## QC: sole-modification Trp C-mannosylation exclusion check

54 glycopeptiforms whose only modification is Trp C-mannosylation are mislabeled as `glycation` throughout the annotation (see Figure1_data_preparation.ipynb and chat discussion). Checked whether any of them appear in the 'Glycopeptiform ID' column of TableS2 -- none do, so no change to this table is needed.

In [ ]:
qc_remove_gpids <- readLines('../2_main_figures/Figure1/data/qc_remove_gpids.txt')
w_sole_exclude_gpids <- readLines('../2_main_figures/Figure1/data/w_sole_exclude_gpids.txt')
stopifnot(length(w_sole_exclude_gpids) == 54)
excluded_gpids <- c(qc_remove_gpids, w_sole_exclude_gpids)
tbl <- read.csv('TableS5.csv', check.names = FALSE)
is_note <- grepl('^Note:', tbl[['Gene']])
stopifnot(sum(is_note) == 1L, which(is_note) == nrow(tbl))
hits <- intersect(unique(tbl[['Glycopeptiform ID']][!is_note]), excluded_gpids)
stopifnot(length(hits) == 0)
cat('No affected gpIDs found in', sum(!is_note), 'data rows of TableS5.csv; footer note verified.\n')

In [ ]:
# Coding-variant check (v202609): 21 glycopeptiforms dropped after the coding-variant screen must not enter any QTL-level object of this table
coding_drop_gpids <- readLines('../2_main_figures/Figure1/data/coding_variant_drop_gpids.txt')
stopifnot(length(coding_drop_gpids) == 21L, !anyDuplicated(coding_drop_gpids))
tbl <- read.csv('TableS5.csv', check.names = FALSE)
is_note <- grepl('^Note:', tbl[['Gene']])
hits <- intersect(unique(tbl[['Glycopeptiform ID']][!is_note]), coding_drop_gpids)
stopifnot(length(hits) == 0)
cat('No coding-variant-dropped gpIDs found in', sum(!is_note), 'data rows of TableS5.csv.\n')
